# Higgs TTS 3 ile okuma listesinden ses veri seti

Bir referans sesle (kendi kaydın ya da hazır bir ses) bir okuma listesindeki her cümleyi seslendirir. Her satır bir WAV olur ve iş ilerledikçe **Google Drive'a kaydedilir**. Colab kopsa da, makine sıfırlansa da **kaldığın yerden devam edersin**.

**Nasıl kullanılır**

1. Runtime → Change runtime type → **L4 GPU** (ya da A100). T4'ün belleği yetmez.
2. **1. hücre**: `MODE = "new"`, Drive klasörünü ve iş adını yaz.
3. Hücreleri sırayla çalıştır. **5–7. hücreler** sesi, ayarları ve okuma listesini sorar. **8. hücre** ilk 3 satırı dinletir. **9. hücre** üretimi başlatır, **10. hücre** ilerlemeyi gösterir.
4. Colab koparsa: **1. hücrede** `MODE = "continue"` seç, aynı klasör ve iş adını yaz, sonra **Runtime → Run all**. Ses, ayarlar ve liste Drive'daki işten okunur, form değerleri yok sayılır.

**Drive'daki iş klasörü**: `manifest.json` (her satırın durumu), `events.jsonl` (her denemenin kaydı), `config.json`, `reference.wav` + `reference.txt`, `reading_list.txt`, `wavs/000001.wav…`, `metadata.csv` (Piper/LJSpeech biçimi `id|metin`), `logs/`.

**Lisans**: `bosonai/higgs-tts-3-4b` araştırma ve ticari olmayan kullanım içindir (içerik üreticilerine ayrı bir izin var, "Higgs Audio" adının anılması gerekir). Bu sesle eğitilen bir modelin ya da veri setinin ticari kullanımı lisansa takılabilir. Başkasının sesini izni olmadan kopyalama.

**Kontrol etiketleri**: Bir satırın başına `<|emotion:sadness|>`, `<|prosody:speed_slow|>`, `<|style:whispering|>` gibi etiketler koyabilirsin; satır içine `<|prosody:pause|>` ya da `<|sfx:laughter|>Haha,` gibi etiketler girebilirsin. Etiketler `metadata.csv`'ye yazılmaz. Tam liste 6. hücrede.

In [ ]:
#@title 1. İş: yeni mi, devam mı? (Drive bağlanır)
MODE = "new" #@param ["new", "continue"]
#@markdown Drive'da işlerin tutulacağı klasör (`MyDrive` altından):
DRIVE_PATH = "voice-me/tts-jobs" #@param {type:"string"}
#@markdown Bu işin adı (klasör adı olur). Devam ederken aynısını yaz:
JOB_NAME = "erdem-okuma-1" #@param {type:"string"}
import json, os, re, sys
from google.colab import drive

drive.mount("/content/drive")
assert re.fullmatch(r"[A-Za-z0-9._-]+", JOB_NAME), "İş adında yalnızca harf, rakam, . _ - kullan"
JOB_DIR = os.path.join("/content/drive/MyDrive", DRIVE_PATH.strip("/"), JOB_NAME)
MANIFEST = os.path.join(JOB_DIR, "manifest.json")
if MODE == "new":
    if os.path.exists(MANIFEST):
        raise SystemExit(f"{JOB_DIR} içinde zaten bir iş var. Devam etmek için MODE = 'continue' seç ya da başka bir iş adı yaz.")
    os.makedirs(JOB_DIR, exist_ok=True)
    print("yeni iş:", JOB_DIR)
else:
    if not os.path.exists(MANIFEST):
        raise SystemExit(f"{JOB_DIR} içinde manifest.json yok. Klasör ve iş adını kontrol et ya da MODE = 'new' seç.")
    summary = json.load(open(MANIFEST, encoding="utf-8"))
    print("devam edilecek iş:", JOB_DIR)
    print("durum:", summary["state"], "| sayılar:", summary["counts"], "| son güncelleme:", summary["updated_at"])
    print("ayarlar:", json.dumps(summary["config"], ensure_ascii=False))
os.makedirs("/content/higgs", exist_ok=True)

In [ ]:
#@title 2. GPU ve disk kontrolü
#@markdown T4 gibi 20 GB'tan az bellekli bir GPU'da yine de denemek için işaretle:
FORCE_SMALL_GPU = False #@param {type:"boolean"}
import shutil, subprocess
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits",
                     shell=True, capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("GPU yok: Runtime → Change runtime type → L4 GPU seç, sonra yeniden çalıştır.")
name, mem_mib = [part.strip() for part in gpu.stdout.strip().split("\n")[0].split(",")]
print(f"GPU: {name}, {int(mem_mib) / 1024:.0f} GB")
if int(mem_mib) < 20000 and not FORCE_SMALL_GPU:
    raise SystemExit("Bu GPU'nun belleği az (en az ~20 GB gerekir). L4 ya da A100 seç, "
                     "ya da FORCE_SMALL_GPU'yu işaretle (büyük ihtimalle bellek yetmez).")
free_gb = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Drive'da boş alan: {free_gb:.1f} GB (1000 satır ≈ 0.5 GB)")

In [ ]:
#@title 3. Kurulum (her yeni makinede bir kez, ~3–4 dk)
import os, shutil, subprocess
os.chdir("/content")
if shutil.which("sgl-omni") is None:
    # outlines-core has no wheel here and builds from source, so it needs Rust.
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
    !uv pip install --system -q --prerelease=allow "sglang-omni==0.2.0" requests librosa soundfile
else:
    print("sglang-omni zaten kurulu")
!hf download bosonai/higgs-tts-3-4b > /dev/null && echo "model indirildi"
import torch
assert torch.cuda.is_available(), "Kurulumdan sonra torch CUDA'yı görmüyor; makineyi sıfırlayıp tekrar dene."
os.makedirs(os.path.join(JOB_DIR, "logs"), exist_ok=True)
!pip freeze > "{JOB_DIR}/logs/pip-freeze.txt"
print("torch", torch.__version__, "| CUDA hazır")

In [ ]:
%%writefile /content/higgs/higgs_common.py
"""Shared helpers for the Higgs TTS dataset notebook and its worker.

Written to /content/higgs/higgs_common.py by the notebook; both the notebook
cells and the detached worker (`higgs_worker.py`) import it, so the server
command, the request shape and the validation rules live in one place.

Truth lives on Drive in three layers: the WAV files, an append-only
`events.jsonl`, and a `manifest.json` snapshot. `load_manifest` replays the
journal over the snapshot and `reconcile` checks it against the files, so a
stale snapshot after a crash costs at most a few re-generated lines.
"""

import datetime
import hashlib
import json
import os
import re
import shutil
import signal
import subprocess
import time

import requests

LOCAL = "/content/higgs"
SERVER_URL = "http://localhost:8000"
MODEL = "bosonai/higgs-tts-3-4b"
SGLANG_OMNI = "0.2.0"
SAMPLE_RATE = 24000
FRAMES_PER_SECOND = 25
SCHEMA_VERSION = 1

# Voice presets. Without a reference clip the model picks a voice from the
# seed, so a preset is "this seed + these tags speaking PRESET_TEXT"; the
# notebook generates that clip once and then clones it for every line, which
# keeps one voice across the whole reading list. F0 is the median pitch
# measured on PRESET_TEXT (librosa.pyin) on an L4 — a label, not a promise.
PRESET_TEXT = (
    "Merhaba, ben Voice Me için hazırlanmış bir deneme sesiyim. Bugün size bu "
    "modelin Türkçeyi ne kadar doğal konuşabildiğini göstereceğim."
)
#
# Findings behind them (L4, sglang-omni 0.2.0): with no tag every seed tried
# came out female (171-210 Hz); `<|prosody:pitch_low|>` gives male voices but
# deep ones (72-112 Hz); the `instructions` field has no effect. A normal male
# voice comes from re-voicing a pitch_low voice with `<|prosody:pitch_high|>`
# ("restage", ~126 Hz; lines cloned from it land ~115 Hz). Shifting a voice up
# with DSP worked on paper but sounded crackly, so no preset uses it. Same seed + same
# text = same clip.
VOICE_PRESETS = {
    "Erkek – normal, seed 38 (~115–126 Hz)": {
        "seed": 38, "tags": "<|prosody:pitch_low|>",
        # This exact text and seed gave the ~126 Hz clip that was picked by
        # ear; other texts move the pitch (the preset text gave ~141 Hz).
        "restage": {"seed": 42, "tags": "<|prosody:pitch_high|>", "text": (
            "Sabah kahvemi içip pencereden dışarı baktım; yağmur yeni dinmişti ve "
            "sokaklar ışıl ışıldı. Bugün güzel bir gün olacak gibi."
        )},
    },
    "Erkek – tok, seed 39 (~112 Hz)": {"seed": 39, "tags": "<|prosody:pitch_low|>"},
    "Erkek – kalın (~80 Hz)": {"seed": 5, "tags": "<|prosody:pitch_low|>"},
    "Kadın – orta (~190 Hz)": {"seed": 10, "tags": ""},
    "Kadın – ince (~210 Hz)": {"seed": 13, "tags": ""},
}


def make_preset_reference(name, out_path):
    """Generate a preset's reference clip at out_path; returns its transcript."""
    preset = VOICE_PRESETS[name]
    params = {"temperature": 0.8, "top_k": 50, "max_new_tokens": 1024}
    with open(out_path, "wb") as f:
        f.write(synth(preset["tags"] + PRESET_TEXT, params, seed=preset["seed"]))
    if "restage" in preset:
        stage1 = out_path + ".stage1.wav"
        os.replace(out_path, stage1)
        restage = preset["restage"]
        text = restage.get("text", PRESET_TEXT)
        with open(out_path, "wb") as f:
            f.write(synth(restage["tags"] + text, params, seed=restage["seed"],
                          ref_path=stage1, ref_text=PRESET_TEXT))
        return text
    return PRESET_TEXT

# The model's 43 control tags (PROMPTING.md in the model repo). Anything else
# inside <|...|> is read aloud or degrades the output.
KNOWN_TAGS = {
    "emotion": {
        "affection", "amusement", "anger", "arousal", "awe", "bitterness",
        "confusion", "contemplation", "contentment", "determination",
        "disgust", "elation", "enthusiasm", "fear", "helplessness", "longing",
        "pride", "relief", "sadness", "shame", "surprise",
    },
    "prosody": {
        "speed_very_slow", "speed_slow", "speed_fast", "speed_very_fast",
        "pitch_low", "pitch_high", "expressive_high", "expressive_low",
        "pause", "long_pause",
    },
    "style": {"singing", "shouting", "whispering"},
    "sfx": {
        "cough", "laughter", "crying", "screaming", "burping", "humming",
        "sigh", "sniff", "sneeze",
    },
}
TAG_RE = re.compile(r"<\|([a-z]+):([a-z_]+)\|>")


def now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def unknown_tags(text):
    """Tags in `text` the model does not know, as "category:tag" strings."""
    bad = [f"{c}:{t}" for c, t in TAG_RE.findall(text) if t not in KNOWN_TAGS.get(c, ())]
    # A stray "<|" that is not a well-formed tag is also a problem.
    if text.count("<|") != len(TAG_RE.findall(text)):
        bad.append("malformed <|...|>")
    return bad


def strip_tags(text):
    """The text as spoken, for metadata.csv: tags removed, spaces collapsed."""
    return re.sub(r"\s+", " ", TAG_RE.sub("", text)).strip()


# ---- Drive I/O: retried, and atomic where it matters ----------------------

def retry_io(fn, *args, tries=5, **kwargs):
    for attempt in range(tries):
        try:
            return fn(*args, **kwargs)
        except OSError:
            if attempt == tries - 1:
                raise
            time.sleep(2 ** attempt)


def _fsync(f):
    f.flush()
    try:
        os.fsync(f.fileno())
    except OSError:
        pass  # best-effort on Drive's FUSE mount


def _write_json_atomic(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
        _fsync(f)
    os.replace(tmp, path)


def write_json_atomic(path, obj):
    retry_io(_write_json_atomic, path, obj)


def _copy_atomic(src, dst):
    part = dst + ".part"
    shutil.copyfile(src, part)
    with open(part, "rb+") as f:
        _fsync(f)
    os.replace(part, dst)


def copy_atomic(src, dst):
    retry_io(_copy_atomic, src, dst)


def _append_line(path, line):
    with open(path, "a", encoding="utf-8") as f:
        f.write(line + "\n")
        _fsync(f)


def append_event(job_dir, event):
    event = {"ts": now(), **event}
    retry_io(_append_line, os.path.join(job_dir, "events.jsonl"), json.dumps(event, ensure_ascii=False))


def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ---- Manifest --------------------------------------------------------------

def count(manifest):
    counts = {"pending": 0, "done": 0, "failed": 0}
    for item in manifest["items"]:
        counts[item["status"]] += 1
    manifest["counts"] = counts
    return counts


def load_manifest(job_dir):
    """manifest.json with events.jsonl replayed over it."""
    manifest = read_json(os.path.join(job_dir, "manifest.json"))
    by_id = {item["id"]: item for item in manifest["items"]}
    events = os.path.join(job_dir, "events.jsonl")
    if os.path.exists(events):
        with open(events, encoding="utf-8") as f:
            for line in f:
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    continue  # a line cut off by a crash
                item = by_id.get(event.get("id"))
                if item is None:
                    continue
                item["attempts"] = max(item["attempts"], event.get("attempt", 0))
                if event["status"] == "done":
                    for key in ("seed", "bytes", "duration_s", "gen_s"):
                        item[key] = event.get(key)
                    item.update(status="done", error=None, finished_at=event["ts"])
                elif event["status"] == "failed":
                    item.update(status="failed", error=event.get("error"), finished_at=event["ts"])
                elif event["status"] == "reset":
                    item.update(status="pending", attempts=0, error=None)
                elif event["status"] == "attempt_failed" and item["status"] != "done":
                    item["error"] = event.get("error")
    count(manifest)
    return manifest


def is_valid_wav(path, expected_bytes=None):
    try:
        size = os.path.getsize(path)
        if expected_bytes and size != expected_bytes:
            return False
        with open(path, "rb") as f:
            return f.read(4) == b"RIFF" and size > 1000
    except OSError:
        return False


def reconcile(job_dir, manifest):
    """Make the manifest agree with the files on Drive. Returns a report."""
    reset, adopted = [], []
    for item in manifest["items"]:
        path = os.path.join(job_dir, item["file"])
        if item["status"] == "done" and not is_valid_wav(path, item.get("bytes")):
            # attempts=0 so the line is redone with the seed it had before.
            item.update(status="pending", attempts=0, error="file missing or incomplete on Drive")
            reset.append(item["id"])
        elif item["status"] != "done" and is_valid_wav(path):
            item.update(status="done", bytes=os.path.getsize(path), error=None)
            adopted.append(item["id"])
    count(manifest)
    return {"reset": reset, "adopted": adopted}


def save_manifest(job_dir, manifest, state=None):
    if state:
        manifest["state"] = state
    manifest["updated_at"] = now()
    count(manifest)
    write_json_atomic(os.path.join(job_dir, "manifest.json"), manifest)


def write_metadata(job_dir, manifest):
    """LJSpeech/Piper style `id|text` for the finished lines."""
    rows = [f"{item['id']}|{strip_tags(item['text'])}" for item in manifest["items"] if item["status"] == "done"]
    path = os.path.join(job_dir, "metadata.csv")

    def write():
        tmp = path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            f.write("\n".join(rows) + ("\n" if rows else ""))
            _fsync(f)
        os.replace(tmp, path)

    retry_io(write)


# ---- Server ----------------------------------------------------------------

def server_script(concurrency=1):
    """The serve command. `concurrency` requests run at once on the GPU;
    cuda_graph_max_bs must cover max_running_requests or the server refuses."""
    return f"""#!/bin/bash
# TORCHDYNAMO_DISABLE: without it startup spends 30+ minutes in torch.compile.
export TORCHDYNAMO_DISABLE=1
exec sgl-omni serve --model-path {MODEL} --port 8000 \\
  --tts_engine.factory.cuda_graph_max_bs {concurrency} \\
  --tts_engine.factory.max_running_requests {concurrency}
"""


def healthy(timeout=3):
    try:
        return requests.get(SERVER_URL + "/health", timeout=timeout).ok
    except requests.RequestException:
        return False


def _server_pid():
    try:
        with open(os.path.join(LOCAL, "server.pid")) as f:
            return int(f.read())
    except (OSError, ValueError):
        return None


def _alive(pid):
    try:
        os.kill(pid, 0)
        return True
    except OSError:
        return False


def stop_server():
    pid = _server_pid()
    if pid and _alive(pid):
        try:
            os.killpg(pid, signal.SIGTERM)
            for _ in range(20):
                if not _alive(pid):
                    break
                time.sleep(1)
            os.killpg(pid, signal.SIGKILL)
        except OSError:
            pass
    # Stage workers are separate processes; make sure the GPU is free.
    # "[s]" keeps the pattern from matching this shell's own command line.
    subprocess.run("pkill -9 -f '[s]gl-omni serve'; pkill -9 -f '[s]glang_omni'", shell=True)
    time.sleep(3)


def _running_concurrency():
    try:
        with open(os.path.join(LOCAL, "server.concurrency")) as f:
            return int(f.read())
    except (OSError, ValueError):
        return 1  # servers started before this file existed ran one at a time


def start_server(log_dir=None, wait_s=900, say=print, concurrency=1):
    """Start the server (stopping any old one) and wait until it is healthy.
    A healthy server started for a different concurrency is restarted."""
    if healthy() and _running_concurrency() == concurrency:
        return True
    stop_server()
    os.makedirs(LOCAL, exist_ok=True)
    script = os.path.join(LOCAL, "server.sh")
    with open(script, "w") as f:
        f.write(server_script(concurrency))
    with open(os.path.join(LOCAL, "server.concurrency"), "w") as f:
        f.write(str(concurrency))
    log_path = os.path.join(LOCAL, "server.log")
    log = open(log_path, "ab")
    proc = subprocess.Popen(["bash", script], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    with open(os.path.join(LOCAL, "server.pid"), "w") as f:
        f.write(str(proc.pid))
    started = time.time()
    while time.time() - started < wait_s:
        if healthy():
            say(f"sunucu hazır ({time.time() - started:.0f} sn)")
            if log_dir:
                try:
                    shutil.copyfile(log_path, os.path.join(log_dir, "server-latest.log"))
                except OSError:
                    pass
            return True
        if proc.poll() is not None:
            break
        time.sleep(5)
    with open(log_path, errors="replace") as f:
        tail = f.read()[-3000:]
    raise RuntimeError("sunucu başlamadı. server.log sonu:\n" + tail)


# ---- Synthesis and validation ------------------------------------------------

# The server's `language` field accepts only these names (sglang-omni 0.2.0
# rejects anything else with HTTP 400, Turkish included). The model itself
# speaks 100+ languages from the text, so every other choice — "tr" too —
# sends no `language` at all and lets the model read the text as it is.
SERVER_LANGUAGES = {
    "en": "English", "zh": "Chinese", "fr": "French", "de": "German", "it": "Italian",
    "ja": "Japanese", "ko": "Korean", "pt": "Portuguese", "ru": "Russian", "es": "Spanish",
}


class RequestRejected(RuntimeError):
    """The server refused the request itself (HTTP 4xx): retrying cannot help."""


def synth(text, config, seed, ref_path=None, ref_text=None, timeout=600):
    """POST one line; returns WAV bytes or raises RuntimeError."""
    body = {
        "input": config.get("prefix_tags", "") + text,
        "temperature": config["temperature"],
        "top_k": config["top_k"],
        "max_new_tokens": config["max_new_tokens"],
        "seed": seed,
    }
    if config.get("top_p"):
        body["top_p"] = config["top_p"]
    if config.get("repetition_penalty"):
        body["repetition_penalty"] = config["repetition_penalty"]
    language = SERVER_LANGUAGES.get(config.get("language") or "auto")
    if language:
        body["language"] = language
    if ref_path:
        body["references"] = [{"audio_path": ref_path, "text": ref_text}]
    response = requests.post(SERVER_URL + "/v1/audio/speech", json=body, timeout=(10, timeout))
    if 400 <= response.status_code < 500:
        raise RequestRejected(f"HTTP {response.status_code}: {response.text[:300]}")
    if not response.ok:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:300]}")
    if response.content[:4] != b"RIFF":
        raise RuntimeError("yanıt WAV değil: " + response.content[:200].decode("utf-8", "replace"))
    return response.content


def read_timeout(text):
    """Seconds to wait for one line: generous, about real time on an L4."""
    return max(120, 8 * len(strip_tags(text)) / 14)


def validate(path, text, config):
    """None if the clip looks right, else the reason. Catches silence,
    truncation and wild length; it cannot catch wrong words."""
    import numpy as np
    import soundfile as sf

    try:
        audio, rate = sf.read(path, dtype="float32", always_2d=True)
    except Exception as error:  # noqa: BLE001 - any decode failure is a bad clip
        return f"okunamadı: {error}"
    audio = audio.mean(axis=1)
    duration = len(audio) / rate
    chars = max(1, len(strip_tags(text)))
    cap = config["max_new_tokens"] / FRAMES_PER_SECOND
    if duration < 0.4:
        return f"çok kısa ({duration:.1f} sn)"
    if duration >= 0.97 * cap:
        return f"kesilmiş görünüyor ({duration:.1f} sn, sınır {cap:.0f} sn)"
    if duration < chars / 28:
        return f"metne göre çok kısa ({duration:.1f} sn, {chars} karakter)"
    if duration > chars / 5 + 1.5:
        return f"metne göre çok uzun ({duration:.1f} sn, {chars} karakter)"
    frame = int(0.02 * rate)
    frames = audio[: len(audio) // frame * frame].reshape(-1, frame)
    rms = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    if 20 * np.log10(rms.mean() + 1e-12) < -45:
        return "neredeyse sessiz"
    silent = rms < 10 ** (-45 / 20)
    if silent.mean() > 0.5:
        return f"%{silent.mean() * 100:.0f} sessizlik"
    trailing = 0
    for is_silent in silent[::-1]:
        if not is_silent:
            break
        trailing += 1
    if trailing * 0.02 > 2.0:
        return f"sonda {trailing * 0.02:.1f} sn sessizlik"
    return None


def wav_duration(path):
    import soundfile as sf

    info = sf.info(path)
    return info.frames / info.samplerate

In [ ]:
%%writefile /content/higgs/higgs_worker.py
"""Detached generation worker: `python3 higgs_worker.py <drive job dir> [concurrency]`.

Runs outside the notebook kernel (started with start_new_session), so a
disconnected browser or an interrupted cell does not stop it. `concurrency`
lines are generated at once (the GPU batches them); each one is on Drive
before it counts as done, `events.jsonl` records every attempt and
`manifest.json` is snapshotted every 60 seconds. Touch /content/higgs/STOP to
stop once the lines in flight finish.
"""

import concurrent.futures
import json
import os
import socket
import sys
import threading
import time
import traceback

sys.path.insert(0, "/content/higgs")
import higgs_common as hc  # noqa: E402

JOB = sys.argv[1]
# Lines generated at once; the server is started with the same capacity.
CONCURRENCY = int(sys.argv[2]) if len(sys.argv) > 2 else 1
LOCAL = hc.LOCAL
TMP = os.path.join(LOCAL, "tmp")
STOP = os.path.join(LOCAL, "STOP")
STATUS = os.path.join(LOCAL, "status.json")
LOCK = os.path.join(JOB, "worker.lock")
os.makedirs(TMP, exist_ok=True)
os.makedirs(os.path.join(JOB, "wavs"), exist_ok=True)
os.makedirs(os.path.join(JOB, "logs"), exist_ok=True)

log_file = open(os.path.join(LOCAL, "worker.log"), "a", encoding="utf-8")


def say(message):
    line = f"{hc.now()} {message}"
    log_file.write(line + "\n")
    log_file.flush()


def status(manifest, **extra):
    data = {"updated": time.time(), "counts": manifest["counts"], "total": len(manifest["items"]), **extra}
    with open(STATUS + ".tmp", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    os.replace(STATUS + ".tmp", STATUS)


def heartbeat():
    try:
        hc.write_json_atomic(LOCK, {"pid": os.getpid(), "host": socket.gethostname(),
                                    "heartbeat": time.time(), "at": hc.now()})
    except OSError:
        pass


def check_frozen(manifest):
    """The frozen inputs on Drive must still be the ones the job started with."""
    ref = os.path.join(JOB, "reference.wav")
    if hc.sha256_file(ref) != manifest["reference"]["sha256"]:
        raise SystemExit("reference.wav değişmiş; işi durduruyorum")
    if hc.sha256_file(os.path.join(JOB, "reading_list.txt")) != manifest["reading_list"]["sha256"]:
        raise SystemExit("reading_list.txt değişmiş; işi durduruyorum")
    if hc.sha256_text(json.dumps(manifest["config"], sort_keys=True)) != manifest["config_sha256"]:
        raise SystemExit("config değişmiş; işi durduruyorum")


class Run:
    """State shared by the generation threads; every change holds `lock`."""

    def __init__(self, manifest, config):
        self.manifest = manifest
        self.config = config
        self.lock = threading.Lock()
        self.server_lock = threading.Lock()
        self.halt = None
        self.consecutive_failures = 0
        self.restarts_in_a_row = 0
        self.done = 0
        self.started = time.time()
        self.in_flight = {}

    def report(self, **extra):
        with self.lock:
            hc.count(self.manifest)
            rate = (time.time() - self.started) / max(1, self.done)
            current = sorted(self.in_flight)
            text = self.manifest["items"][self.in_flight[current[0]]]["text"][:120] if current else ""
            status(self.manifest, current=", ".join(current), text=text, rate=rate,
                   done_this_run=self.done, concurrency=CONCURRENCY, **extra)

    def ensure_server(self):
        """Restart the server if it is down; one thread does it, the rest wait."""
        with self.server_lock:
            if hc.healthy():
                return
            with self.lock:
                self.restarts_in_a_row += 1
                restarts = self.restarts_in_a_row
            if restarts > 3:
                self.halt = "sunucu üst üste 3 kez yeniden başlatılamadı"
                return
            say("sunucu yanıt vermiyor; yeniden başlatıyorum")
            time.sleep(5 * 3 ** (restarts - 1))
            try:
                hc.start_server(log_dir=os.path.join(JOB, "logs"), say=say, concurrency=CONCURRENCY)
            except RuntimeError as start_error:
                say(str(start_error)[-500:])


def process(run, index, item, ref_local, ref_text):
    """Generate one line: retry with a new seed while the clip fails validation;
    server trouble retries the same seed and does not use up an attempt."""
    config, max_attempts = run.config, run.config["max_attempts"]
    error, server_errors = None, 0
    while item["attempts"] < max_attempts and not run.halt:
        seed = config["base_seed"] + index + 1000 * item["attempts"]
        tmp = os.path.join(TMP, item["id"] + ".wav")
        t0 = time.time()
        try:
            wav = hc.synth(item["text"], config, seed, ref_local, ref_text, timeout=hc.read_timeout(item["text"]))
            with open(tmp, "wb") as f:
                f.write(wav)
            problem = hc.validate(tmp, item["text"], config)
            if problem:
                raise ValueError(problem)
            duration = round(hc.wav_duration(tmp), 2)
            hc.copy_atomic(tmp, os.path.join(JOB, item["file"]))
            gen_s = round(time.time() - t0, 2)
            with run.lock:
                item["attempts"] += 1
                item.update(status="done", seed=seed, bytes=len(wav), duration_s=duration,
                            gen_s=gen_s, error=None, finished_at=hc.now())
                hc.append_event(JOB, {"id": item["id"], "attempt": item["attempts"], "seed": seed,
                                      "status": "done", "bytes": len(wav), "duration_s": duration, "gen_s": gen_s})
                run.consecutive_failures, run.restarts_in_a_row = 0, 0
                run.done += 1
            return
        except hc.RequestRejected as rejected:  # a settings problem, the same for every line
            run.halt = f"sunucu isteği reddetti, ayarları kontrol et: {rejected}"
            return
        except ValueError as invalid:  # the clip came back but looks wrong: next seed
            error = str(invalid)
            with run.lock:
                item["attempts"] += 1
                say(f"{item['id']} deneme {item['attempts']}: {error}")
                hc.append_event(JOB, {"id": item["id"], "attempt": item["attempts"], "seed": seed,
                                      "status": "attempt_failed", "error": error})
        except Exception as failure:  # noqa: BLE001 - server/network/Drive trouble
            error = f"{type(failure).__name__}: {failure}"[:500]
            server_errors += 1
            say(f"{item['id']} sunucu/ağ hatası {server_errors}: {error}")
            run.ensure_server()
            if server_errors % 3 == 0:  # bounded: a line that keeps breaking the server still ends
                with run.lock:
                    item["attempts"] += 1
                    hc.append_event(JOB, {"id": item["id"], "attempt": item["attempts"], "seed": seed,
                                          "status": "attempt_failed", "error": error})
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)
    if run.halt:
        return
    with run.lock:
        item.update(status="failed", error=error, finished_at=hc.now())
        hc.append_event(JOB, {"id": item["id"], "attempt": item["attempts"], "status": "failed", "error": error})
        run.consecutive_failures += 1
        if run.consecutive_failures >= 10:
            run.halt = "üst üste 10 satır başarısız; ayarları kontrol et"


def main():
    manifest = hc.load_manifest(JOB)
    check_frozen(manifest)
    report = hc.reconcile(JOB, manifest)
    if report["reset"] or report["adopted"]:
        say(f"uzlaştırma: {len(report['reset'])} satır yeniden üretilecek, {len(report['adopted'])} dosya benimsendi")
    hc.save_manifest(JOB, manifest, state="running")
    config = manifest["config"]
    ref_text = manifest["reference"]["text"]
    # The server reads the reference from local disk, never through Drive's FUSE.
    ref_local = os.path.join(LOCAL, "reference.wav")
    hc.copy_atomic(os.path.join(JOB, "reference.wav"), ref_local)

    hc.start_server(log_dir=os.path.join(JOB, "logs"), say=say, concurrency=CONCURRENCY)
    say(f"başladı: {manifest['counts']}, aynı anda {CONCURRENCY} satır")
    run = Run(manifest, config)
    heartbeat()
    # The seed index comes from the line's id (1-based line number in the
    # frozen list), not its position here, so a job resumed from a subset of
    # the items still gives every line the seed it would have had.
    queue = iter([(int(item["id"]) - 1, item) for item in manifest["items"] if item["status"] == "pending"])
    futures, stopping, last_snapshot = {}, False, time.time()

    def submit_more(pool):
        while len(futures) < CONCURRENCY and not stopping and not run.halt:
            nxt = next(queue, None)
            if nxt is None:
                return
            index, item = nxt
            with run.lock:
                run.in_flight[item["id"]] = index
            futures[pool.submit(process, run, index, item, ref_local, ref_text)] = item["id"]

    with concurrent.futures.ThreadPoolExecutor(CONCURRENCY) as pool:
        submit_more(pool)
        while futures:
            finished, _ = concurrent.futures.wait(futures, timeout=5,
                                                  return_when=concurrent.futures.FIRST_COMPLETED)
            for future in finished:
                item_id = futures.pop(future)
                with run.lock:
                    run.in_flight.pop(item_id, None)
                try:
                    future.result()
                except Exception:  # noqa: BLE001 - a bug in one line must not lose the others
                    say(f"{item_id} beklenmeyen hata:\n" + traceback.format_exc())
            if not stopping and os.path.exists(STOP):
                stopping = True
                say("STOP dosyası bulundu; işteki satırlar bitince duruyorum")
            submit_more(pool)
            run.report()
            if time.time() - last_snapshot > 60:
                with run.lock:
                    hc.save_manifest(JOB, manifest)
                    hc.write_metadata(JOB, manifest)
                heartbeat()
                last_snapshot = time.time()

    with run.lock:
        counts = hc.count(manifest)
        hc.save_manifest(JOB, manifest, state="complete" if counts["pending"] == 0 else "halted")
        hc.write_metadata(JOB, manifest)
    run.report(finished=True, halted=run.halt)
    if run.halt:
        raise SystemExit(run.halt)
    say(f"bitti: {counts}")


if __name__ == "__main__":
    try:
        main()
    except SystemExit as stop:
        say(f"durdu: {stop}")
        try:
            with open(STATUS, encoding="utf-8") as f:
                data = json.load(f)
        except (OSError, ValueError):
            data = {}
        data.update(finished=True, halted=str(stop))
        with open(STATUS, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False)
    except Exception:  # noqa: BLE001
        say("çöktü:\n" + traceback.format_exc())
        raise
    finally:
        try:
            import shutil
            shutil.copyfile(os.path.join(LOCAL, "worker.log"), os.path.join(JOB, "logs", "worker-latest.log"))
            os.remove(LOCK)
        except OSError:
            pass

In [ ]:
#@title 4. Model sunucusunu başlat (~2 dk; ilk seferde birkaç dk daha)
#@markdown Aynı anda kaç satır üretilsin. L4'te ölçülen (satır başına): 1 → 4,2 sn, 8 → 0,68 sn, 16 → 0,41 sn, 32 → 0,25 sn. Ses aynı kalır. 64 L4'te belleğe sığmadı; bellek hatası görürsen düşür.
CONCURRENCY = 32 #@param [1, 4, 8, 16, 32] {type:"raw"}
import sys
sys.path.insert(0, "/content/higgs")
import importlib, higgs_common as hc
importlib.reload(hc)
hc.start_server(log_dir=os.path.join(JOB_DIR, "logs"), concurrency=CONCURRENCY)
# One throwaway request absorbs the first-use kernel builds.
hc.synth("Merhaba.", {"temperature": 0.8, "top_k": 50, "max_new_tokens": 256}, seed=1)
print("ısınma tamam")

In [ ]:
#@title 5. Ses: hazır ses ya da kendi referans kaydın (yalnızca yeni işte)
#@markdown **Hazır sesler** seed'den üretilir ve her satırda o ses kopyalanır. Etiketteki Hz, ölçülen ortalama ses perdesidir.
VOICE = "Erkek – normal, seed 38 (~115–126 Hz)" #@param ["Kendi kaydım", "Erkek – normal, seed 38 (~115–126 Hz)", "Erkek – tok, seed 39 (~112 Hz)", "Erkek – kalın (~80 Hz)", "Kadın – orta (~190 Hz)", "Kadın – ince (~210 Hz)"]
#@markdown ---
#@markdown **Kendi kaydım** seçiliyse: Drive'daki dosyanın yolu (boşsa yükleme penceresi açılır). WAV, MP3, FLAC, M4A olur; 5–15 sn ideal.
REF_DRIVE_PATH = "" #@param {type:"string"}
#@markdown Kayıtta tam olarak ne söylendiği (zorunlu; klonlama kalitesini çok etkiler):
REF_TRANSCRIPT = "" #@param {type:"string"}
import shutil, subprocess
import soundfile as sf
from IPython.display import Audio, display

if MODE == "continue":
    print("devam modu: referans Drive'daki işten okunuyor")
    display(Audio(os.path.join(JOB_DIR, "reference.wav")))
else:
    raw = "/content/higgs/ref_raw"
    if VOICE == "Kendi kaydım":
        assert REF_TRANSCRIPT.strip(), "REF_TRANSCRIPT boş: kayıtta ne söylendiğini yaz"
        if REF_DRIVE_PATH.strip():
            src = REF_DRIVE_PATH if REF_DRIVE_PATH.startswith("/") else os.path.join("/content/drive/MyDrive", REF_DRIVE_PATH)
            shutil.copyfile(src, raw)
        else:
            from google.colab import files
            uploaded = files.upload()
            assert uploaded, "dosya yüklenmedi"
            open(raw, "wb").write(next(iter(uploaded.values())))
        ref_text = REF_TRANSCRIPT.strip()
        voice_info = {"kind": "upload", "label": VOICE}
    else:
        ref_text = hc.make_preset_reference(VOICE, raw)
        voice_info = {"kind": "preset", "label": VOICE, **hc.VOICE_PRESETS[VOICE]}
    # 24 kHz mono 16-bit, leading/trailing silence trimmed.
    ref_local = "/content/higgs/reference.wav"
    trim = "silenceremove=start_periods=1:start_threshold=-45dB,areverse,silenceremove=start_periods=1:start_threshold=-45dB,areverse"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, "-af", trim,
                    "-ac", "1", "-ar", "24000", "-sample_fmt", "s16", ref_local], check=True)
    duration = sf.info(ref_local).duration
    print(f"referans: {duration:.1f} sn")
    if duration > 30:
        raise SystemExit("Referans 30 sn'den uzun; 5–15 sn'lik bir parça kullan.")
    if not 5 <= duration <= 15:
        print("uyarı: 5–15 sn arası en iyi sonucu verir")
    hc.copy_atomic(ref_local, os.path.join(JOB_DIR, "reference.wav"))
    with open(os.path.join(JOB_DIR, "reference.txt"), "w", encoding="utf-8") as f:
        f.write(ref_text + "\n")
    REFERENCE = {"sha256": hc.sha256_file(ref_local), "duration_s": round(duration, 2), "text": ref_text, "voice": voice_info}
    display(Audio(ref_local))

In [ ]:
#@title 6. Model ayarları (yalnızca yeni işte)
#@markdown Dil: model dili metinden anlar; Türkçe için `auto` ya da `tr` aynıdır. Yalnızca en, zh, fr, de, it, ja, ko, pt, ru, es sunucuya ayrıca bildirilir (sunucu başka dil adı kabul etmiyor).
LANGUAGE = "auto" #@param ["auto", "tr", "en", "de", "fr", "es", "it", "pt", "ru", "ja", "ko", "zh"]
TEMPERATURE = 0.8 #@param {type:"slider", min:0.1, max:1.5, step:0.05}
TOP_K = 50 #@param {type:"integer"}
#@markdown 0 = kullanma:
TOP_P = 0 #@param {type:"number"}
REPETITION_PENALTY = 0 #@param {type:"number"}
#@markdown 1024 ≈ en fazla 40 sn ses. Uzun satırlar için artır:
MAX_NEW_TOKENS = 1024 #@param {type:"integer"}
#@markdown Her satırın seed'i = BASE_SEED + satır sırası (tekrar üretilebilir):
BASE_SEED = 42 #@param {type:"integer"}
#@markdown Her satırın başına eklenecek etiketler, ör. `<|prosody:speed_slow|>` (boş = yok):
PREFIX_TAGS = "" #@param {type:"string"}
#@markdown Doğrulamadan geçemeyen satır kaç kez (farklı seed'le) denensin:
MAX_ATTEMPTS = 4 #@param {type:"integer"}
#@markdown `#` ile başlayan satırları atla:
SKIP_COMMENT_LINES = True #@param {type:"boolean"}

print("Kullanılabilir etiketler:")
for category, tags in hc.KNOWN_TAGS.items():
    print(f"  {category}: " + ", ".join(f"<|{category}:{t}|>" for t in sorted(tags)))
if MODE == "continue":
    print("\ndevam modu: ayarlar Drive'daki işten okunuyor; bu formdaki değerler kullanılmaz")
else:
    assert not hc.unknown_tags(PREFIX_TAGS), f"bilinmeyen etiket: {hc.unknown_tags(PREFIX_TAGS)}"
    CONFIG = {
        "model": hc.MODEL, "sglang_omni": hc.SGLANG_OMNI, "language": LANGUAGE,
        "temperature": TEMPERATURE, "top_k": TOP_K, "top_p": TOP_P or None,
        "repetition_penalty": REPETITION_PENALTY or None, "max_new_tokens": MAX_NEW_TOKENS,
        "base_seed": BASE_SEED, "prefix_tags": PREFIX_TAGS, "max_attempts": MAX_ATTEMPTS,
        "seed_policy": "base_seed + index + 1000 * attempt", "skip_comment_lines": SKIP_COMMENT_LINES,
    }
    print("\nayarlar:", json.dumps(CONFIG, ensure_ascii=False))

In [ ]:
#@title 7. Okuma listesi: her satır bir cümle (yalnızca yeni işte)
#@markdown Drive'daki .txt dosyasının yolu (boşsa yükleme penceresi açılır):
LIST_DRIVE_PATH = "" #@param {type:"string"}
import unicodedata

if MODE == "continue":
    print("devam modu: liste Drive'daki işten okunuyor")
else:
    if LIST_DRIVE_PATH.strip():
        src = LIST_DRIVE_PATH if LIST_DRIVE_PATH.startswith("/") else os.path.join("/content/drive/MyDrive", LIST_DRIVE_PATH)
        raw_bytes = open(src, "rb").read()
    else:
        from google.colab import files
        uploaded = files.upload()
        assert uploaded, "dosya yüklenmedi"
        raw_bytes = next(iter(uploaded.values()))
    text = unicodedata.normalize("NFC", raw_bytes.decode("utf-8-sig"))
    items, seen = [], {}
    for line_no, line in enumerate(text.splitlines(), 1):
        line = line.strip()
        if not line or (CONFIG["skip_comment_lines"] and line.startswith("#")):
            continue
        bad = hc.unknown_tags(line)
        if bad:
            print(f"uyarı satır {line_no}: bilinmeyen etiket {bad}")
        if len(hc.strip_tags(line)) > 350:
            print(f"uyarı satır {line_no}: çok uzun ({len(line)} karakter), {CONFIG['max_new_tokens'] // 25} sn sınırını aşabilir")
        if line in seen:
            print(f"uyarı satır {line_no}: {seen[line]}. satırın tekrarı")
        seen.setdefault(line, line_no)
        n = len(items) + 1
        items.append({"id": f"{n:06d}", "line_no": line_no, "text": line, "status": "pending", "attempts": 0,
                      "seed": None, "file": f"wavs/{n:06d}.wav", "bytes": None, "duration_s": None,
                      "gen_s": None, "error": None, "finished_at": None})
    assert items, "listede okunacak satır yok"
    frozen = os.path.join(JOB_DIR, "reading_list.txt")
    with open(frozen, "w", encoding="utf-8") as f:
        f.write("\n".join(item["text"] for item in items) + "\n")
    hc.write_json_atomic(os.path.join(JOB_DIR, "config.json"), CONFIG)
    manifest = {
        "schema_version": hc.SCHEMA_VERSION, "job_name": JOB_NAME, "created_at": hc.now(), "updated_at": hc.now(),
        "state": "pending", "config": CONFIG, "config_sha256": hc.sha256_text(json.dumps(CONFIG, sort_keys=True)),
        "reference": REFERENCE, "reading_list": {"sha256": hc.sha256_file(frozen), "n_items": len(items)},
        "counts": {}, "items": items,
    }
    hc.save_manifest(JOB_DIR, manifest)
    est_h = sum(len(hc.strip_tags(i["text"])) for i in items) / 14 / 3600
    print(f"{len(items)} satır kaydedildi. Tahmini süre: ~{est_h:.1f} saat (L4'te ≈ gerçek zaman)")

In [ ]:
#@title 8. Ön dinleme: ilk 3 satır (işe yazılmaz)
#@markdown Beğenmezsen 5–6. hücrelere dönüp ayarı değiştir; iş henüz başlamadı. (Yeni işte değişiklikten sonra 7. hücreyi de tekrar çalıştır.)
PREVIEW_LINES = 3 #@param {type:"integer"}
from IPython.display import Audio, display
manifest = hc.load_manifest(JOB_DIR)
CONFIG = manifest["config"]
ref_local = "/content/higgs/reference.wav"
hc.copy_atomic(os.path.join(JOB_DIR, "reference.wav"), ref_local)
for index, item in enumerate(manifest["items"][:PREVIEW_LINES]):
    wav = hc.synth(item["text"], CONFIG, CONFIG["base_seed"] + index, ref_local, manifest["reference"]["text"],
                   timeout=hc.read_timeout(item["text"]))
    path = f"/content/higgs/preview_{item['id']}.wav"
    open(path, "wb").write(wav)
    problem = hc.validate(path, item["text"], CONFIG)
    print(f"{item['id']}: {item['text'][:100]}" + (f"  ⚠ {problem}" if problem else ""))
    display(Audio(path))

In [ ]:
#@title 9. Üretimi başlat (arka planda çalışır; bu hücre hemen biter)
#@markdown Devam ederken daha önce başarısız olan satırlar da yeniden denensin:
RETRY_FAILED = False #@param {type:"boolean"}
import socket, subprocess, time
lock = os.path.join(JOB_DIR, "worker.lock")
if os.path.exists(lock):
    held = json.load(open(lock))
    # A lock left by a crashed worker on this same machine is stale at once.
    crashed_here = held.get("host") == socket.gethostname() and not os.path.exists(f"/proc/{held.get('pid')}")
    if not crashed_here and time.time() - held.get("heartbeat", 0) < 120:
        raise SystemExit("Bu iş şu an başka bir yerde çalışıyor gibi (son sinyal 2 dk'dan yeni). "
                         "Aynı işi iki makinede birden çalıştırma.")
if RETRY_FAILED:
    manifest = hc.load_manifest(JOB_DIR)
    for item in manifest["items"]:
        if item["status"] == "failed":
            hc.append_event(JOB_DIR, {"id": item["id"], "status": "reset"})
if os.path.exists("/content/higgs/STOP"):
    os.remove("/content/higgs/STOP")
if subprocess.run("pgrep -f '[h]iggs_worker.py'", shell=True, capture_output=True).returncode == 0:
    print("çalışan bir üretim zaten var; 10. hücreden izle")
else:
    subprocess.Popen(["python3", "/content/higgs/higgs_worker.py", JOB_DIR, str(CONCURRENCY)],
                     stdout=open("/content/higgs/worker.out", "ab"), stderr=subprocess.STDOUT, start_new_session=True)
    print("başladı. 10. hücreden izle; tarayıcıyı kapatsan da makine açık kaldıkça devam eder.")

In [ ]:
#@title 10. İlerleme (durdurmak üretimi durdurmaz; istediğin zaman tekrar çalıştır)
#@markdown Her yeni tamamlanan satırı dinlet:
PLAY_EACH = False #@param {type:"boolean"}
import time
from IPython.display import Audio, clear_output, display
status_path = "/content/higgs/status.json"
last_played = None
while True:
    try:
        st = json.load(open(status_path, encoding="utf-8"))
    except (OSError, ValueError):
        st = {}
    clear_output(wait=True)
    counts, total = st.get("counts", {}), st.get("total", 0)
    done = counts.get("done", 0)
    bar = "█" * int(30 * done / max(1, total))
    print(f"[{bar:<30}] {done}/{total}  başarısız: {counts.get('failed', 0)}")
    rate = st.get("rate")
    if rate and st.get("done_this_run"):
        left = counts.get("pending", 0) * rate
        print(f"satır başına ~{rate:.1f} sn, kalan ~{left / 3600:.1f} saat")
    if st.get("current"):
        ids = st["current"].split(", ")
        print(f"şimdi ({len(ids)} satır aynı anda): {ids[0]}…{ids[-1]}  ör. {st.get('text', '')}")
    print("\nson günlük satırları:")
    print(subprocess.run("tail -n 6 /content/higgs/worker.log", shell=True, capture_output=True, text=True).stdout)
    if PLAY_EACH and st.get("last_done") and st["last_done"] != last_played:
        last_played = st["last_done"]
        path = os.path.join(JOB_DIR, "wavs", last_played + ".wav")
        if os.path.exists(path):
            display(Audio(path))
    running = subprocess.run("pgrep -f '[h]iggs_worker.py'", shell=True, capture_output=True).returncode == 0
    if st.get("finished") or not running:
        print("üretim çalışmıyor." + (f" sebep: {st['halted']}" if st.get("halted") else ""), "11–12. hücrelere geç.")
        break
    time.sleep(10)

In [ ]:
#@title 11. Durdur (o anki satır bitince durur; sonra 9. hücreyle devam edebilirsin)
#@markdown "Run all" bu hücreyi de çalıştırır; yanlışlıkla durmasın diye yalnızca bu kutu işaretliyken durdurur:
STOP_NOW = False #@param {type:"boolean"}
if STOP_NOW:
    open("/content/higgs/STOP", "w").close()
    print("durdurma isteği bırakıldı; o anki satır bitince durur")
else:
    print("durdurmak için STOP_NOW'u işaretleyip bu hücreyi tekrar çalıştır")

In [ ]:
#@title 12. Özet, metadata.csv ve zip
#@markdown Piper eğitimi için 22050 Hz kopyası da çıkar:
EXPORT_22050 = True #@param {type:"boolean"}
import shutil, subprocess, zipfile
if subprocess.run("pgrep -f '[h]iggs_worker.py'", shell=True, capture_output=True).returncode == 0:
    raise SystemExit("Üretim hâlâ çalışıyor; bitmesini bekle ya da önce 11. hücreyle durdur. "
                     "(\"Run all\" ile buraya geldiysen sorun yok, üretim devam ediyor.)")
manifest = hc.load_manifest(JOB_DIR)
hc.reconcile(JOB_DIR, manifest)
hc.save_manifest(JOB_DIR, manifest)
hc.write_metadata(JOB_DIR, manifest)
counts = manifest["counts"]
total_s = sum(i["duration_s"] or 0 for i in manifest["items"] if i["status"] == "done")
print(f"bitti: {counts['done']}, başarısız: {counts['failed']}, bekleyen: {counts['pending']} | toplam ses: {total_s / 60:.1f} dk")
failed = [i for i in manifest["items"] if i["status"] == "failed"]
if failed:
    with open(os.path.join(JOB_DIR, "failed.txt"), "w", encoding="utf-8") as f:
        for item in failed:
            f.write(f"{item['id']}\t{item['error']}\t{item['text']}\n")
    print(f"{len(failed)} başarısız satır failed.txt'ye yazıldı; 9. hücrede RETRY_FAILED ile tekrar denenebilir")
out = f"/content/higgs/{JOB_NAME}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(JOB_DIR, "metadata.csv"), "metadata.csv")
    for item in manifest["items"]:
        if item["status"] != "done":
            continue
        src = os.path.join(JOB_DIR, item["file"])
        if EXPORT_22050:
            dst = f"/content/higgs/tmp/{item['id']}.wav"
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-ar", "22050", dst], check=True)
            z.write(dst, f"wavs/{item['id']}.wav")
            os.remove(dst)
        else:
            z.write(src, f"wavs/{item['id']}.wav")
shutil.copyfile(out, os.path.join(JOB_DIR, os.path.basename(out)))
print("zip Drive'a kopyalandı:", os.path.join(JOB_DIR, os.path.basename(out)))
drive.flush_and_unmount()
print("Drive senkronize edildi ve ayrıldı (tekrar kullanmak için 1. hücreyi çalıştır)")